# 03 데이터 전처리

원본 아파트 매매 실거래가와 전월세 실거래가 데이터를 자치구 × 월 단위로 정리한다.

전처리 목표는 다음과 같다.

- 거래금액, 보증금, 월세 금액을 수치형으로 변환
- 계약년월을 월 기준으로 정리
- 월세 금액이 0이면 전세, 0보다 크면 월세로 구분
- 자치구·월별 평균 매매가, 평균 전세 보증금, 거래량, 월세 비중 계산
- 매매 데이터와 전월세 데이터를 자치구·월 기준으로 병합
- `data/processed/monthly_merged.csv` 생성

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()


if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sale = pd.read_csv(RAW_DIR / "sale_all.csv", low_memory=False)
rent = pd.read_csv(RAW_DIR / "rent_all.csv", low_memory=False)

print("프로젝트 루트:", ROOT)
print("raw 데이터 폴더:", RAW_DIR)
print("processed 데이터 폴더:", PROCESSED_DIR)
print("sale shape:", sale.shape)
print("rent shape:", rent.shape)

gu_map = {
    11110:'종로구',11140:'중구',11170:'용산구',11200:'성동구',11215:'광진구',
    11230:'동대문구',11260:'중랑구',11290:'성북구',11305:'강북구',11320:'도봉구',
    11350:'노원구',11380:'은평구',11410:'서대문구',11440:'마포구',11470:'양천구',
    11500:'강서구',11530:'구로구',11545:'금천구',11560:'영등포구',11590:'동작구',
    11620:'관악구',11650:'서초구',11680:'강남구',11710:'송파구',11740:'강동구'
}

def num_clean(s):
    return pd.to_numeric(
        s.astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )

def clean_sale(df):
    d = df.copy()
    d['sggCd'] = pd.to_numeric(d['sggCd'], errors='coerce').astype('Int64')
    d['gu'] = d['sggCd'].map(gu_map)
    d['contract_month'] = d['dealYear'].astype(str) + '-' + d['dealMonth'].astype(str).str.zfill(2)
    d['sale_price'] = num_clean(d['dealAmount'])
    d['area_m2'] = pd.to_numeric(d['excluUseAr'], errors='coerce')
    d['sale_price_per_m2'] = d['sale_price'] / d['area_m2']
    d['age'] = pd.to_numeric(d['dealYear'], errors='coerce') - pd.to_numeric(d['buildYear'], errors='coerce')
    return d

def clean_rent(df):
    d = df.copy()
    d['sggCd'] = pd.to_numeric(d['sggCd'], errors='coerce').astype('Int64')
    d['gu'] = d['sggCd'].map(gu_map)
    d['contract_month'] = d['dealYear'].astype(str) + '-' + d['dealMonth'].astype(str).str.zfill(2)
    d['deposit_num'] = num_clean(d['deposit'])
    d['monthly_rent_num'] = num_clean(d['monthlyRent'])
    d['area_m2'] = pd.to_numeric(d['excluUseAr'], errors='coerce')
    d['deposit_per_m2'] = d['deposit_num'] / d['area_m2']
    d['rent_type'] = np.where(
        d['monthly_rent_num'].fillna(0) == 0, 
        'jeonse', 'monthly')
    d['age'] = pd.to_numeric(d['dealYear'], errors='coerce') - pd.to_numeric(d['buildYear'], errors='coerce')
    return d

sale_clean = clean_sale(sale)
rent_clean = clean_rent(rent)

sale_monthly = sale_clean.groupby(['sggCd','gu','contract_month'], dropna=False).agg(
    avg_sale_price=('sale_price','mean'),
    med_sale_price=('sale_price','median'),
    avg_sale_price_per_m2=('sale_price_per_m2','mean'),
    med_sale_price_per_m2=('sale_price_per_m2','median'),
    sale_count=('sale_price','size')
).reset_index()

jeonse_monthly = rent_clean[rent_clean['rent_type'] == 'jeonse'].groupby(['sggCd','gu','contract_month'], dropna=False).agg(
    avg_jeonse_deposit=('deposit_num','mean'),
    med_jeonse_deposit=('deposit_num','median'),
    avg_jeonse_deposit_per_m2=('deposit_per_m2','mean'),
    med_jeonse_deposit_per_m2=('deposit_per_m2','median'),
    jeonse_count=('deposit_num','size')
).reset_index()

rent_monthly = rent_clean.groupby(['sggCd','gu','contract_month'], dropna=False).agg(
    total_rent_count=('deposit_num','size'),
    monthly_count=('rent_type', lambda x: int((x == 'monthly').sum())),
    jeonse_count_from_rent=('rent_type', lambda x: int((x == 'jeonse').sum()))
).reset_index()

rent_monthly['monthly_ratio'] = rent_monthly['monthly_count'] / rent_monthly['total_rent_count']

monthly_merged = sale_monthly.merge(
    jeonse_monthly,
    on=['sggCd','gu','contract_month'],
    how='inner'
).merge(
    rent_monthly[['sggCd','gu','contract_month','total_rent_count','monthly_count','monthly_ratio']],
    on=['sggCd','gu','contract_month'],
    how='left'
)

monthly_merged = monthly_merged.sort_values(['sggCd','contract_month']).reset_index(drop=True)

monthly_merged['jeonse_rate'] = monthly_merged['avg_jeonse_deposit_per_m2'] / monthly_merged['avg_sale_price_per_m2']
monthly_merged['gap_rate'] = 1 - monthly_merged['jeonse_rate']

monthly_merged['sale_growth_1m'] = monthly_merged.groupby('sggCd')['avg_sale_price_per_m2'].pct_change()
monthly_merged['jeonse_growth_1m'] = monthly_merged.groupby('sggCd')['avg_jeonse_deposit_per_m2'].pct_change()
monthly_merged['growth_gap_1m'] = monthly_merged['sale_growth_1m'] - monthly_merged['jeonse_growth_1m']
monthly_merged['sale_volume_growth_1m'] = monthly_merged.groupby('sggCd')['sale_count'].pct_change()
monthly_merged['rent_volume_growth_1m'] = monthly_merged.groupby('sggCd')['total_rent_count'].pct_change()

monthly_merged.to_csv(PROCESSED_DIR / "monthly_merged.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", PROCESSED_DIR / "monthly_merged.csv")
print("monthly_merged shape:", monthly_merged.shape)

프로젝트 루트: /Users/min/RealEstate-Risk-Radar
raw 데이터 폴더: /Users/min/RealEstate-Risk-Radar/data/raw
processed 데이터 폴더: /Users/min/RealEstate-Risk-Radar/data/processed
sale shape: (188054, 22)
rent shape: (747197, 27)
저장 완료: /Users/min/RealEstate-Risk-Radar/data/processed/monthly_merged.csv
monthly_merged shape: (900, 23)


In [5]:
monthly_merged.info()
monthly_merged.head()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   sggCd                      900 non-null    Int64  
 1   gu                         900 non-null    str    
 2   contract_month             900 non-null    str    
 3   avg_sale_price             900 non-null    float64
 4   med_sale_price             900 non-null    float64
 5   avg_sale_price_per_m2      900 non-null    float64
 6   med_sale_price_per_m2      900 non-null    float64
 7   sale_count                 900 non-null    int64  
 8   avg_jeonse_deposit         900 non-null    float64
 9   med_jeonse_deposit         900 non-null    float64
 10  avg_jeonse_deposit_per_m2  900 non-null    float64
 11  med_jeonse_deposit_per_m2  900 non-null    float64
 12  jeonse_count               900 non-null    int64  
 13  total_rent_count           900 non-null    int64  
 14  month

,sggCd,gu,contract_month,avg_sale_price,med_sale_price,avg_sale_price_per_m2,med_sale_price_per_m2,sale_count,avg_jeonse_deposit,med_jeonse_deposit,...,total_rent_count,monthly_count,monthly_ratio,jeonse_rate,gap_rate,sale_growth_1m,jeonse_growth_1m,growth_gap_1m,sale_volume_growth_1m,rent_volume_growth_1m
0,11110,종로구,2023-05,97098.529412,93495.0,1315.438781,1180.755760,34,59292.695238,59000.0,...,208,103,0.495192,0.600280,0.399720,NaN,NaN,NaN,NaN,NaN
1,11110,종로구,2023-06,125363.333333,118250.0,1382.048126,1282.217351,30,65890.121951,63000.0,...,160,78,0.487500,0.620928,0.379072,0.050637,0.086775,-0.036139,-0.117647,-0.230769
2,11110,종로구,2023-07,119448.181818,127500.0,1427.666364,1283.479442,22,62944.702970,55000.0,...,174,73,0.419540,0.587078,0.412922,0.033008,-0.023307,0.056315,-0.266667,0.087500
3,11110,종로구,2023-08,133262.000000,137225.0,1582.590432,1401.408319,40,59248.738739,48000.0,...,213,102,0.478873,0.532480,0.467520,0.108516,0.005424,0.103092,0.818182,0.224138
4,11110,종로구,2023-09,102655.208333,94250.0,1321.018893,1148.612117,48,57560.235849,51000.0,...,217,111,0.511521,0.578222,0.421778,-0.165281,-0.093574,-0.071707,0.200000,0.018779
